# Handling Semi Structured Data 

In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json

import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    ArrayType
)

from common_utils.logging import get_logger
from common_utils.generic_functions import rename_columns, safe_cast
from common_utils.scdfunctions import scd_type_1


# ============================================================
# 0. LOGGER
# ============================================================

logger = get_logger("sales_orders_silver")


# ============================================================
# 1. READ CONFIG
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

logger.info("Reading Silver configuration from %s", config_path)

with open(config_path, "r") as f:
    config = json.load(f)


# ============================================================
# 2. SOURCE
# ============================================================

source_table = config["source"]["source_table"]
source_name = config["source"]["source_name"]


# ============================================================
# 3. PRODUCT SCHEMA
# ============================================================

product_schema = StructType([
    StructField("id", StringType()),
    StructField("name", StringType()),
    StructField("price", StringType()),
    StructField("curr", StringType()),
    StructField("qty", StringType()),
    StructField("unit", StringType())
])


# ============================================================
# 4. PROMOTION SCHEMA
# ============================================================

promotion_schema = StructType([
    StructField("promo_id", StringType()),
    StructField("promo_item", StringType()),
    StructField("promo_disc", DoubleType()),
    StructField("promo_qty", StringType())
])


# ============================================================
# 5. SALES ORDER SCHEMA
# ============================================================

sales_order_schema = StructType([
    StructField("order_number", StringType()),
    StructField("customer_id", StringType()),
    StructField("customer_name", StringType()),
    StructField("number_of_line_items", StringType()),

    # Handles values such as 1.564627663E9
    StructField("order_datetime", DoubleType()),

    StructField(
        "ordered_products",
        ArrayType(product_schema)
    ),

    StructField(
        "promo_info",
        ArrayType(promotion_schema)
    ),

    StructField(
        "clicked_items",
        ArrayType(
            ArrayType(StringType())
        )
    )
])


# ============================================================
# 6. PARSE JSON DATA
# ============================================================

def parse_sales_orders(df):

    logger.info(
        "[%s] Parsing json_data column",
        source_name
    )

    df = (
        df
        .withColumn("data",F.from_json( F.col("json_data"),sales_order_schema )
        )
        .select("data.*")
    )

    logger.info("[%s] json_data successfully parsed",source_name)
    return df


# ============================================================
# 7. SALES ORDERS
# ============================================================

def transform_sales_orders(df, config):

    logger.info(
        "[%s] Transforming sales_orders",
        source_name
    )

    df = rename_columns(
        df,
        config["tables"]["sales_orders"].get(
            "rename_columns",
            {}
        )
    )

    df = (
        df
        .withColumn(
            "order_timestamp",
            F.from_unixtime( F.col("order_datetime").cast("long")).cast("timestamp")
        )
        .select("order_number","customer_id","customer_name","number_of_line_items","order_timestamp"))

    df = safe_cast(
        df,
        config["tables"]["sales_orders"].get("cast_columns",{})
    )

    df = df.select(
        *config["tables"]["sales_orders"]["select_columns"]
    )

    logger.info("[%s] sales_orders transformation completed", source_name)
    df = df.dropDuplicates()

    return df


# ============================================================
# 8. ORDER PRODUCTS
# ============================================================

def transform_order_products(df, config):

    logger.info(
        "[%s] Transforming order_products",
        source_name
    )

    df = (
        df
        .withColumn(
            "ordered_product",
            F.explode_outer(
                F.col("ordered_products")
            )
        )
        .select(
            "customer_id",
            "customer_name",
            "order_number",
            F.col("ordered_product.id").alias("id"),
            F.col("ordered_product.name").alias("product_name"),
            F.col("ordered_product.price").alias("price"),
            F.col("ordered_product.curr").alias("curr"),
            F.col("ordered_product.qty").alias("qty"),
            F.col("ordered_product.unit").alias("unit")
        )
        .filter(
            F.col("id").isNotNull()
        )
    )

    df = safe_cast(
        df,
        config["tables"]["order_products"].get(
            "cast_columns",
            {}
        )
    )

    df = df.select(
        *config["tables"]["order_products"]["select_columns"]
    )

    logger.info(
        "[%s] order_products transformation completed",
        source_name
    )

    df = df.dropDuplicates()

    return df


# ============================================================
# 9. PROMOTIONS
# ============================================================

def transform_promotions(df, config):

    logger.info( "[%s] Transforming promotions",source_name)

    df = (
        df
        .withColumn(
            "promotion",
            F.explode_outer(
                F.col("promo_info")
            )
        )
        .select(
            "customer_id",
            "order_number",
            F.col("promotion.promo_id").alias("promo_id"),
            F.col("promotion.promo_item").alias("promo_product_id"),
            F.col("promotion.promo_disc").alias("discount"),
            F.col("promotion.promo_qty").alias("promo_quantity")
        )
        .filter(
            F.col("promo_id").isNotNull()
        )
    )

    df = safe_cast(
        df,
        config["tables"]["promotions"].get(
            "cast_columns",
            {}
        )
    )

    df = df.select(*config["tables"]["promotions"]["select_columns"])

    logger.info( "[%s] promotions transformation completed", source_name)

    df = df.dropDuplicates()

    return df


# ============================================================
# 10. CLICKED ITEMS
# ============================================================

def transform_clicked_items(df, config):

    logger.info("[%s] Transforming clicked_items", source_name)

    df = (
        df
        .withColumn(
            "clicked_item",
            F.explode_outer(
                F.col("clicked_items")
            )
        )
        .select(
            "customer_id",
            "customer_name",
            "order_number",
            F.col("clicked_item")
            .getItem(0)
            .alias("product_id"),
            F.col("clicked_item")
            .getItem(1)
            .alias("score")
        )
        .filter(
            F.col("product_id").isNotNull()
        )
    )

    df = safe_cast(
        df,
        config["tables"]["clicked_items"].get(
            "cast_columns",
            {}
        )
    )

    df = df.select( *config["tables"]["clicked_items"]["select_columns"])

    logger.info("[%s] clicked_items transformation completed", source_name)

    df = df.dropDuplicates()

    return df


# ============================================================
# 11. READ BRONZE
# ============================================================

logger.info(
    "[%s] Reading from Bronze table %s",
    source_name,
    source_table
)

df = spark.read.table(source_table)

logger.info("[%s] Successfully read Bronze table",source_name)


# ============================================================
# 12. PARSE NESTED JSON
# ============================================================

df = parse_sales_orders(df)


# ============================================================
# 13. TRANSFORM FOUR SILVER TABLES
# ============================================================

sales_orders = transform_sales_orders(df,config)

order_products = transform_order_products(df,config)

promotions = transform_promotions(
    df,
    config
)

clicked_items = transform_clicked_items(df,config)


# ============================================================
# 14. WRITE / SCD TYPE 1
# ============================================================

for table_name, transformed_df in {
    "sales_orders": sales_orders,
    "order_products": order_products,
    "promotions": promotions,
    "clicked_items": clicked_items
}.items():

    table_config = config["tables"][table_name]

    target_table = table_config["target_table"]

    scd_config = table_config.get(
        "scd",
        {}
    )

    logger.info("[%s] Processing target table %s",table_name,target_table)

    # --------------------------------------------------------
    # Target exists
    # --------------------------------------------------------

    if spark.catalog.tableExists(target_table):

        if scd_config.get("type") == 1:

            logger.info("[%s] Starting SCD Type 1",table_name )

            scd_type_1(
                spark=spark,
                source_df=transformed_df,
                target_table=target_table,
                key_columns=scd_config.get(
                    "key_columns",
                    []
                )
            )

        else:

            raise ValueError(
                f"Unsupported SCD type "
                f"[{scd_config.get('type')}] "
                f"for table [{table_name}]"
            )

    # --------------------------------------------------------
    # Initial load
    # --------------------------------------------------------

    else:

        logger.info("[%s] Target does not exist - creating table",table_name)

        spark.sql("""
            CREATE SCHEMA IF NOT EXISTS
            retaildataplatform.silver
        """)

        (
            transformed_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )

        logger.info("[%s] Silver table created successfully",table_name)


# ============================================================
# 15. COMPLETION
# ============================================================

logger.info("[%s] All Silver tables processed successfully",source_name)
     